# Finalize annotations

**Pinned Environment:** [`envs/sc-scvi.yaml`](../.../envs/sc-scvi.yaml)

* `seurat_labels` > `cell_type`
* Clarify neuron_labels

In [1]:
from pathlib import Path
import sys
import os
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
import seaborn as sns

In [2]:
# from projects/drg
adata_path_seurat = "/home/workspace/DRG/sc-spatial-gpu_ide_transfer/projects/drg/data/h5ad/export_04/04c_gamma_subtypes/neurons_seurat.h5ad"

plot_out_dir = "/home/workspace/DRG/spatial_mouse_drg_outputs/Artis/plots"
if not os.path.exists (plot_out_dir):
    os.makedirs(plot_out_dir)

## Prepare data

In [4]:
adata = sc.read_h5ad(adata_path_seurat)
adata.X = adata.layers["log1p"].copy()

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/home/workspace/DRG/sc-spatial-gpu_ide_transfer/projects/drg/data/h5ad/export_04/04c_gamma_subtypes/neurons_seurat.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

#### Create `reporter_status` column

In [ ]:
# Define and apply conditional mapping
conditions = [
    adata.obs["EGFP_dTomato_dual_hi"] == True,
    (adata.obs["EGFP_Seq1_hi"] == True) & (adata.obs["EGFP_dTomato_dual_hi"] == False),
    (adata.obs["dTomato_Seq2_hi"] == True)
    & (adata.obs["EGFP_dTomato_dual_hi"] == False),
]

choices = ["EGFP_dTomato_dual_hi", "EGFP_Seq1_hi", "dTomato_Seq2_hi"]
adata.obs["reporter_status"] = np.select(conditions, choices, default="Negative")
adata.obs["reporter_status"] = adata.obs["reporter_status"].astype("category")

print(adata.obs.reporter_status.value_counts())

#### Create `innervation_pattern` column

In [ ]:
innervation_map = {
    "EGFP_dTomato_dual_hi": "Double-positive",
    "EGFP_Seq1_hi": "Single-positive",
    "dTomato_Seq2_hi": "Single-positive",
    "Negative": "Negative",
}

adata.obs["innervation_pattern"] = adata.obs["reporter_status"].map(innervation_map)

adata.obs["innervation_pattern"] = adata.obs["innervation_pattern"].astype("category")
pattern_order = ["Double-positive", "Single-positive", "Negative"]
adata.obs["innervation_pattern"] = adata.obs[
    "innervation_pattern"
].cat.reorder_categories(pattern_order)

print(adata.obs.innervation_pattern.value_counts())

### CGRP-gamma subclusters

In [ ]:
gamma_adata = adata[adata.obs["cell_type"] == "CGRP-Gamma"].copy() # Seurat-transferred labels

sc.pp.neighbors(gamma_adata, use_rep="X_scVI_seurat_neuron")
sc.tl.umap(gamma_adata)
sc.tl.leiden(gamma_adata, resolution=0.1, key_added="gamma_subclusters")

# Identify G1 vs G2 based on ranked Chrna3 mean expression
gamma_adata.obs["Chrna3_temp"] = gamma_adata[:, "Chrna3"].X.toarray().flatten()

cluster_expr = gamma_adata.obs.groupby("gamma_subclusters", observed=False)[
    "Chrna3_temp"
].mean()
g2_cluster = cluster_expr.idxmax()
g1_cluster = cluster_expr.idxmin()

del gamma_adata.obs["Chrna3_temp"] # Clean temporary column

label_map = {g1_cluster: "CGRP-Gamma 1", g2_cluster: "CGRP-Gamma 2"}
gamma_adata.obs["neuron_labels"] = gamma_adata.obs["gamma_subclusters"].map(label_map)

# Update parent anndata with new labels
adata.obs["neuron_labels"] = adata.obs["cell_type"].astype(str)
adata.obs.update(gamma_adata.obs[["neuron_labels"]])
adata.obs["neuron_labels"] = adata.obs["neuron_labels"].astype("category")

# Map subclusters onto adata

In [ ]:
adata.obs["neuron_label"] = adata.obs["cell_type"].astype(str)

# From Ginty lab 2024 dataset - Cluster 1 = Chrna3 high (G2), Cluster 0 = Adra2a high (G1)
gamma_map = {"1": "CGRP-Gamma 2", "0": "CGRP-Gamma 1"}
gamma_adata.obs["refined_label"] = gamma_adata.obs["gamma_subclusters"].map(gamma_map)

# Update the main object's neuron_label column
adata.obs.set_index(adata.obs.index, inplace=True)
adata.obs.update(
    gamma_adata.obs[["refined_label"]].rename(columns={"refined_label": "neuron_label"})
) # only overwrites the cells present in gamma_adata

adata.obs["neuron_label"] = adata.obs["neuron_label"].astype("category")

In [ ]:
assign_cell_type_colors(adata, key="neuron_label")

# Export

In [ ]:
adata_path = os.path.join(output_dir, "neurons-final-labels.h5ad")
adata.write_h5ad(adata_path, compression="gzip")